# <font color='blue'>Data Science Academy</font>
# <font color='blue'>Deep Learning Para Aplicações de IA com PyTorch e Lightning</font>
## <font color='blue'>Projeto 2</font>
### <font color='blue'>Simulações Financeiras com Deep Reinforcement Learning</font>

## Instalando e Carregando Pacotes

In [ ]:
# Versão da Linguagem Python
from platform import python_version
print('Versão da Linguagem Python Usada Neste Jupyter Notebook:', python_version())

In [ ]:
# Para atualizar um pacote, execute o comando abaixo no terminal ou prompt de comando:
# pip install -U nome_pacote

# Para instalar a versão exata de um pacote, execute o comando abaixo no terminal ou prompt de comando:
# !pip install nome_pacote==versão_desejada

# Depois de instalar ou atualizar o pacote, reinicie o jupyter notebook.

# Instala o pacote watermark. 
# Esse pacote é usado para gravar as versões de outros pacotes usados neste jupyter notebook.
!pip install -q -U watermark

In [ ]:
!pip install -q yfinance

In [ ]:
# Imports
import torch
import sklearn
import numpy as np
import pandas as pd
import torch.optim as optim
import torch.nn as nn
import yfinance as yf
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from itertools import accumulate
from torch.distributions import Normal

In [ ]:
# Versões dos pacotes usados neste jupyter notebook
%reload_ext watermark
%watermark -a "Data Science Academy" --iversions

In [ ]:
# Define o device
torch.manual_seed(0)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

## Módulo Para Extração dos Dados

In [ ]:
# Define a função dsa_get_close_price, que recebe um identificador de ação (ticker), 
# um período padrão (30 dias) e um intervalo padrão (30 minutos), e retorna uma série do pandas
def dsa_get_close_price(ticker, period = '30d', interval = '30m') -> pd.Series:
    
    # Retorna a série de preços de fechamento ('Close') para o ticker fornecido, usando o Yahoo Finance (yf), 
    # com o período e intervalo especificados
    return yf.Ticker(ticker).history(period = period, interval = interval)['Close']

## Módulo Para Normalização dos Dados

In [ ]:
# Define a função dsa_normalize que normaliza uma série numpy utilizando o MinMaxScaler, 
# aceitando um parâmetro para a frequência de treinamento (padrão 0.5)
def dsa_normalize(series: np.ndarray, train_freq=0.5) -> np.ndarray: 
    
    # Redimensiona a série para um array 2D com uma coluna (necessário para o MinMaxScaler)
    series = series.reshape((len(series), 1))
    
    # Inicializa o MinMaxScaler e o ajusta usando a parte inicial da série definida pela train_freq
    scaler = MinMaxScaler().fit(series[:int(len(series)*train_freq)])
    
    # Aplica a transformação do MinMaxScaler na série inteira
    series = scaler.transform(series)
    
    # Converte a série de volta para um array 1D
    series = series.flatten()
    
    # Retorna a série normalizada
    return series

## Módulo Para Calcular o Índice de Força Relativa (RSI) 

O Índice de Força Relativa (RSI, do inglês "Relative Strength Index") é um indicador de momentum utilizado na análise técnica que mede a magnitude das mudanças recentes no preço para avaliar condições de sobrecompra ou sobrevenda em um ativo financeiro, como ações, moedas ou commodities. O RSI é um dos indicadores técnicos mais conhecidos e utilizados por traders e investidores.

Usaremos Índice de Força Relativa como recursos.

In [ ]:
# Define a função dsa_get_relative_strength_index, que calcula o Índice de Força Relativa (RSI) 
# para uma série pandas de preços, com um período padrão de 14 dias
def dsa_get_relative_strength_index(price: pd.Series, periods = 14) -> pd.Series:
    
    # Calcula a mudança diária de preço e preenche quaisquer NaNs com 0
    change = price.diff().fillna(0)
    
    # Obtém os ganhos (valores positivos da mudança) e os limita ao mínimo de 0 (ignorando perdas)
    gain = change.clip(lower = 0)
    
    # Obtém as perdas (valores negativos da mudança) e os transforma em valores positivos
    loss = -1 * change.clip(upper = 0)
    
    # Calcula a média móvel exponencial (EMA) dos ganhos, usando o período especificado
    ma_gain = gain.ewm(com = periods - 1, adjust = True, min_periods = periods).mean()
    
    # Calcula a média móvel exponencial (EMA) das perdas, usando o período especificado
    ma_loss = loss.ewm(com = periods - 1, adjust = True, min_periods = periods).mean()
    
    # Calcula a relação de força (RS) como a média dos ganhos dividida pela média das perdas
    rs = ma_gain / ma_loss
    
    # Calcula o RSI usando a fórmula padrão
    rsi = 100 - (100/(1+rs))
    
    # Preenche quaisquer NaNs com 50, que é um valor neutro para o RSI
    rsi = rsi.fillna(50)
    
    # Normaliza o RSI para ficar entre -1 e 1
    rsi = ((rsi/100) * 2) - 1
    
    # Retorna a série do RSI normalizado
    return rsi

## Módulo Para Calcular a Convergência e Divergência de Médias Móveis

A Convergência e Divergência de Médias Móveis (MACD, do inglês "Moving Average Convergence Divergence") é um dos indicadores técnicos mais populares usados na análise técnica do mercado financeiro. Este indicador ajuda a identificar a direção da tendência de um ativo, bem como possíveis reversões de tendência. 

Usaremos o MACD como recurso.

Componentes do MACD:

**Linha MACD**: É a diferença entre duas médias móveis exponenciais (EMAs) do preço de um ativo, tipicamente as EMAs de 12 e 26 períodos.

**Linha de Sinal**: É uma média móvel exponencial da linha MACD, geralmente calculada para 9 períodos.

**Histograma MACD**: Representa a diferença entre a linha MACD e a linha de sinal. O histograma é positivo quando a linha MACD está acima da linha de sinal (convergência) e negativo quando está abaixo (divergência).

**Convergência**: Ocorre quando a linha MACD se aproxima da linha de sinal. Isso geralmente é interpretado como um sinal de que o momento (momentum) do ativo está enfraquecendo e pode indicar uma potencial reversão ou desaceleração da tendência atual.

**Divergência**: Acontece quando a linha MACD se afasta da linha de sinal. Uma divergência pode ser um sinal de que o momento está se fortalecendo e que a tendência atual pode continuar.

In [ ]:
# Define a função dsa_moving_avg_conv_div que calcula a Convergência e Divergência de Médias Móveis (MACD) 
# normalizada de uma série de preços
def dsa_moving_avg_conv_div(price: pd.Series, train_freq = 0.5) -> np.ndarray:
    
    # Calcula a média móvel exponencial (EMA) de curto prazo (12 períodos) dos preços
    exp1 = price.ewm(span = 12, adjust = False).mean()
    
    # Calcula a média móvel exponencial (EMA) de longo prazo (26 períodos) dos preços
    exp2 = price.ewm(span = 26, adjust = False).mean()
    
    # Calcula a linha MACD como a diferença entre as EMAs de curto e longo prazo
    macd_line = exp1 - exp2
    
    # Calcula a linha de sinal do MACD como a EMA de 9 períodos da linha MACD
    signal_line = macd_line.ewm(span = 9, adjust = False).mean()
    
    # Calcula o histograma do MACD como a diferença entre a linha MACD e a linha de sinal
    h = macd_line - signal_line
    
    # Normaliza o histograma do MACD (h) usando a função dsa_normalize e o ajusta para um intervalo de -1 a 1
    norm_h = (dsa_normalize(h.values, train_freq = train_freq) * 2) - 1
    
    # Retorna o histograma do MACD normalizado
    return norm_h

## Módulo Para Formatar os Dados com os Novos Recursos

Por fim concatenamos o preço normalizado, o RSI e o MCAD ao vetor de estado e também retornamos o preço de fechamento.

Usaremos dados da Apple (AAPL).

In [ ]:
# Define a função dsa_get_states para obter dados de estado (preço, RSI, MACD) para uma ação específica 
# em um intervalo e período definidos
def dsa_get_states(ticker = 'AAPL', period = '2y', interval = '1h', train_freq = 0.5):
    
    # Obtém a série de preços de fechamento para o ticker especificado, período e intervalo
    close_price = dsa_get_close_price(ticker, period, interval)
    
    # Normaliza os preços de fechamento
    normalized_price = dsa_normalize(close_price.values, train_freq)
    
    # Calcula o Índice de Força Relativa (RSI) para os preços de fechamento
    rsi = dsa_get_relative_strength_index(close_price)
    
    # Calcula a Convergência e Divergência de Médias Móveis (MACD) para os preços de fechamento
    macd = dsa_moving_avg_conv_div(close_price)
    
    # Concatena os dados normalizados, RSI e MACD em um único array e transpõe para o formato correto
    states = np.concatenate(([normalized_price], [rsi], [macd]), axis=0).T
    
    # Retorna a série de preços de fechamento e o array de estados combinados
    return close_price, states

## Define a Política

> Esta política permite ao agente considerar os custos de transação. Está dividida em 2 partes.

## Módulo (Classe) Para o Modelo de Rede Neural

In [ ]:
# Define a classe DSARedeNeural, que é um subtipo do módulo nn.Module do PyTorch
class DSARedeNeural(nn.Module):
    
    # O construtor inicializa o módulo com espaços de observação e ação específicos, e um desvio padrão 
    # para a camada de saída
    def __init__(self, observation_space = 8, action_space = 1, out_layer_std = 3e-3) -> None:
        
        # Chama o construtor da superclasse nn.Module
        super(DSARedeNeural, self).__init__()
        
        # Define o espaço de ação como um atributo da classe
        self.action_space = action_space
        
        # Cria uma camada linear (fully connected) com entrada da soma do espaço de observação e ação, 
        # e saída do espaço de ação
        self.fc_out = nn.Linear((observation_space + self.action_space), action_space)
        
        # Inicializa os pesos da camada de saída com uma distribuição normal, tendo média 0 e desvio padrão 
        # definido por out_layer_std
        self.fc_out.weight.data.normal_(0, out_layer_std)

    # Define o método forward, que especifica como os dados passam pelo modelo
    def forward(self, x, prev_action = None) -> torch.Tensor:
        
        # Se a ação anterior não for fornecida, cria um tensor de zeros com o formato adequado e o coloca 
        # no dispositivo correto (GPU ou CPU)
        if prev_action is None: 
            prev_action = torch.Tensor(np.zeros((x.shape[0],self.action_space))).to(device)         
        
        # Concatena a entrada x com a ação anterior e coloca o resultado no dispositivo correto
        x = torch.cat((x, prev_action.to(device)), dim=1).to(device) 
        
        # Passa a entrada concatenada através da camada linear (fully connected)
        x = self.fc_out(x)
        
        # Retorna a saída do modelo
        return x

## Módulo Para as Ações do Agente com Base na Política

A política é definida como uma densidade de probabilidade normal sobre uma ação escalar de valor real
$
\pi_{\theta, \epsilon} ({a}| \mathbf{s}) = \frac{1}{\epsilon \sqrt{2\pi}} e^{\left(-\frac{\left({a} - \mu_\theta(\mathbf{s}) \right)^2}{2 \epsilon^2}\right)}
$

onde $\mu_\theta(\mathbf{s})$ é o aproximador da função definido acima e $\epsilon \geq 0$ é uma taxa de exploração decrescente
$
\epsilon \leftarrow \max{(\lambda_\epsilon \epsilon, \epsilon_{min})}
$

Definimos a política estocástica como:

In [ ]:
# Define a função dsa_act_policy que seleciona ações de acordo com uma política baseada em uma rede neural, 
# dado um estado e uma ação anterior opcional
def dsa_act_policy(net: nn.Module, state: np.ndarray, prev_action=None, epsilon=0):
    
    # Converte o estado de numpy para tensor do PyTorch, altera seu tipo para float, adiciona uma dimensão 
    # extra e move para o dispositivo (GPU ou CPU)
    state = torch.from_numpy(state).float().unsqueeze(0).to(device)
    
    # Calcula a média (mean) passando o estado e a ação anterior pela rede neural e move o resultado para 
    # o dispositivo
    mean = net.forward(state, prev_action).to(device)
    
    # Aplica a função tangente hiperbólica na média para obter valores entre -1 e 1
    mean = torch.tanh(mean)
    
    # Define o desvio padrão (std) como o máximo entre o valor de epsilon e um valor mínimo para evitar 
    # divisão por zero
    std = max(epsilon, 1e-8)
    
    # Cria uma distribuição normal com a média e o desvio padrão especificados
    dist = Normal(mean, std) 
    
    # Amostra uma ação da distribuição criada
    action = dist.sample() 
    
    # Retorna a ação, limitando seus valores entre -1 e 1, convertendo para numpy e achatando o array, 
    # e o logaritmo da probabilidade da ação na distribuição
    return torch.clamp(action, -1, 1).cpu().numpy().flatten(), dist.log_prob(action).to(device)

## Módulos Para Calcular as Recompensas

Primeiro definimos algumas funções auxiliares para calcular a função de recompensa.
Essas funções ajudarão a calcular os retornos logarítmicos multiplicativos líquidos
$
{r}^{net}_t = \log{\left(  \frac{p_t}{p_{t-1}} \right)} {a}_{t-1}  - \lambda_c || {a}_{t-1} - {a}'_{t-1} || 
$
onde $\lambda_c \in [0,1]$ é a fração do custo de transação e
$
{a}'_t = \frac{a_{t-1} \frac{p_t}{p_{t-1}}}{a_{t-1} y_t + 1}
$

A primeira função calcula o retorno líquido do ativo:

In [ ]:
# Define a função dsa_asset_return que calcula o retorno líquido de um ativo considerando as posições 
# novas e antigas, preços novos e antigos, e custos de transação
def dsa_asset_return(position_new, 
                     position_old, 
                     price_new, 
                     price_old, 
                     transaction_fraction = 0.0002) -> np.ndarray:
    
    # Calcula o retorno bruto como o produto da posição nova e o logaritmo da razão entre o preço novo e 
    # o preço antigo, adicionando um pequeno valor epsilon para evitar divisão por zero
    gross_return = position_new * np.log((price_new + \
                                          float(np.finfo(np.float32).eps)) / (price_old + \
                                                                              float(np.finfo(np.float32).eps)))
    
    # Calcula o retorno líquido subtraindo os custos de transação (calculados pela função dsa_transaction_costs) 
    # do retorno bruto
    net_return = gross_return - dsa_transaction_costs(position_new, 
                                                      position_old, 
                                                      price_new, 
                                                      price_old, 
                                                      transaction_fraction)
    
    # Retorna o retorno líquido
    return net_return

A segunda função calcula os custos de transação.

In [ ]:
# Define a função dsa_transaction_costs para calcular os custos de transação com base nas posições e 
# preços novos e antigos, e na fração de transação
def dsa_transaction_costs(position_new, 
                          position_old, 
                          price_new, 
                          price_old, 
                          transaction_fraction = 0.0002):
    
    # Calcula as mudanças de preço, adicionando um pequeno valor epsilon (finfo) para evitar divisão por zero
    price_changes = ((price_new + float(np.finfo(np.float32).eps)) / (price_old + float(np.finfo(np.float32).eps))) 
    
    # Calcula o peso como resultado da mudança de preço, ajustando a posição antiga com base na variação do preço
    weight_as_result_of_price_change = (position_old * price_changes) / ((position_old * (price_changes-1)) + 1)
    
    # Determina o quanto é necessário reequilibrar (rebalancear) comparando a nova posição com o peso ajustado 
    # pela mudança de preço
    required_rebalance = np.abs(position_new - weight_as_result_of_price_change)
    
    # Calcula os custos de transação como uma fração do valor necessário para reequilibrar
    t_costs = transaction_fraction * required_rebalance
    
    # Retorna os custos de transação
    return t_costs

## Módulo Para Definir o Ambiente

Então definimos o ambiente. A função de recompensa final usa a variação sobre os retornos
$
\sigma^2(r^{net}_i | i=t-L+1,...,t)=\sigma^2_{L}(r^{net}_t)
$
(onde $L=60$) como um termo de risco na função de recompensa final
$
r_t = {r}^{net}_t - \lambda_\sigma \sigma^2_{L}({r}^{net}_t)
$
onde $\lambda_\sigma \geq 0$ é um termo de sensibilidade ao risco.

In [ ]:
# Define a classe DSATradeEnvironment, um ambiente de negociação para simulação e treinamento de algoritmos 
# de aprendizado de máquina
class DSATradeEnvironment():

    # Construtor da classe, inicializa o ambiente com os estados, preços, e parâmetros configuráveis 
    # para a simulação
    def __init__(self, 
                 states, 
                 prices,
                 transaction_fraction = 0.002, 
                 num_prev_observations = 10, 
                 std_lookback = 60, 
                 std_factor = 0.1) -> None:
        
        # Armazena os preços das ações
        self.prices = prices  
        
        # Define a fração de custo de transação
        self.transaction_fraction = transaction_fraction  
        
        # Armazena os estados (dados de mercado)
        self.states = states  
        
        # Índice atual na série temporal
        self.current_index = 0  
        
        # Posição inicial 
        self.position = np.zeros((1,))  
        
        # Armazena os retornos
        self.returns = np.zeros((len(prices),))  
        
        # Fator de ajuste para o cálculo de risco
        self.std_factor = std_factor  
        
        # Janela de observação para o cálculo de risco
        self.std_lookback = std_lookback  
        
        # Cria um histórico de observações iniciais com base no estado atual
        self.observations = np.array([self.states[self.current_index] for _ in range(num_prev_observations)]) 

    # Função para reiniciar o ambiente e retornar o estado inicial
    def reset(self) -> np.ndarray:
        
        # Reinicia o índice atual
        self.current_index = 0  
        
        # Zera as posições
        self.position.fill(0)  
        
        # Recria o histórico de observações iniciais
        self.observations = np.array([self.states[self.current_index] for _ in range(len(self.observations))]) 
        
        # Retorna o estado inicial
        return self.state()  

    # Função para obter o estado atual do ambiente
    def state(self) -> np.ndarray:
        
        # Retorna o vetor de estado achatado
        return self.observations.flatten()  

    # Função para calcular a recompensa com base na ação tomada
    def reward_function(self, action) -> float: 
        
        # Calcula o retorno do ativo com base na ação atual e anterior, e os preços
        ret = dsa_asset_return(position_new = action, 
                               position_old = self.position, 
                               price_new = self.prices[self.current_index],
                               price_old = self.prices[self.current_index-1],
                               transaction_fraction = self.transaction_fraction)
        
        # Armazena o retorno
        self.returns[self.current_index-1] = ret  
        
        # Calcula a punição por risco com base no desvio padrão dos retornos
        # Ou seja, punimos o agente se tomar uma decisão que reduz o retorno líquido
        risk_punishment = np.std(self.returns[max(0, (self.current_index - self.std_lookback)):self.current_index])
        
        # Retorna a recompensa
        return np.sum(ret) - (self.std_factor * risk_punishment)  

    # Função para atualizar o estado do ambiente com base na ação tomada
    def step(self, action):

        # Atualiza o índice atual
        self.current_index += 1  
        
        # Verifica se a ação está no intervalo [-1, 1]
        assert all(action <= 1.) and all(action >= -1.)  
        
        # Se alcançou o fim dos estados, retorna o estado atual, recompensa 0 e sinaliza que terminou
        if self.current_index >= len(self.states): 
            return self.state(), 0, True, {} 
        
        # Calcula a recompensa
        reward = self.reward_function(action)  
        
        # Atualiza a posição com a ação tomada
        self.position = action  
        
        # Atualiza o histórico de observações com o novo estado
        self.observations = np.concatenate((self.observations[1:], [self.states[self.current_index]]), axis=0)
        
        # Retorna o estado atualizado, recompensa, sinal de continuação e informações adicionais vazias
        return self.state(), reward, False, {}  

## Modelo de Reinforcement Learning

Primeiramente definimos a função de otimização, que consiste em redefinir os gradientes da rede, realizar uma passagem para trás e uma etapa de otimização utilizando o otimizador Adam.

## Módulo de Otimização da Rede Neural

In [ ]:
# Define a função dsa_optimize, que realiza a otimização de uma rede neural usando 
# o otimizador Adam e uma determinada função de perda
def dsa_optimize(optimizer: optim.Adam, loss: torch.Tensor) -> None: 
    
    # Zera os gradientes acumulados no otimizador para evitar a acumulação entre iterações
    optimizer.zero_grad()  

    # Realiza a retropropagação (backpropagation) para calcular os gradientes da função de perda em 
    # relação aos parâmetros
    loss.backward()  

    # Realiza um passo de otimização (atualiza os parâmetros da rede neural com base nos gradientes calculados)
    optimizer.step()  

## Módulo da Função de Perda

A seguir definimos a função de perda de política, que é definida como o gradiente da medida de desempenho $J$ é calculado usando o teorema do gradiente de política em minilotes $[t_s, t_e]$ ao seguir a política $\pi_{\theta, \epsilon}$
$
\nabla_\theta J(\pi_{\theta, \epsilon})_{[t_{s},t_{e}]} = \mathbb{E}_{\pi_{\theta, \epsilon}} \left[ \sum_{t=t_s + 1}^{t_e} r_t \nabla_\theta \log \pi_{\theta, \epsilon}(a_t | s_t) \right]
$

In [ ]:
# Define a função dsa_get_policy_loss, que calcula a perda da política para algoritmos de aprendizado por reforço
def dsa_get_policy_loss(rewards: list, log_probs: list) -> torch.Tensor:
    
    # Converte a lista de recompensas para um tensor do PyTorch e move para o dispositivo (GPU ou CPU)
    r = torch.FloatTensor(rewards).to(device)  

    # Empilha e comprime a lista de logaritmos das probabilidades e move para o dispositivo
    log_probs = torch.stack(log_probs).squeeze().to(device)  

    # Calcula a perda da política como o produto dos log_probs e recompensas, inverte o sinal e soma
    policy_loss = torch.mul(log_probs, r).mul(-1).sum().to(device)  

    # Retorna o tensor de perda da política
    return policy_loss  

## Módulo de Treinamento do Agente

In [ ]:
# Define a função dsa_treina_agente para treinar um agente de aprendizado por reforço usando uma 
# rede neural e um ambiente específico
def dsa_executa_agente(policy_network: torch.nn.Module, 
                       env, 
                       act, 
                       alpha = 1e-3, 
                       discount_factor = 0, 
                       weight_decay = 1e-2, 
                       exploration_rate = 1, 
                       exploration_decay = 0.8, 
                       exploration_min = 0.1, 
                       num_episodes = 1000, 
                       train = True):

    # Inicializa o otimizador Adam com os parâmetros da rede neural, taxa de aprendizado e decaimento de peso
    optimizer = optim.Adam(policy_network.parameters(), lr = alpha, weight_decay = weight_decay)
    
    # Inicializa listas para armazenar o histórico de recompensas e ações
    reward_history, action_history = [], []  

    if not train:
        
        # Se não estiver em treinamento, zera a taxa de exploração
        exploration_min, exploration_rate = 0, 0 
        
        # Coloca a rede neural em modo de avaliação
        policy_network.eval()  
        
    else:
        
        # Coloca a rede neural em modo de treinamento
        policy_network.train()  

    # Itera sobre o número definido de episódios
    for n in range(num_episodes):  
        
        # Reinicia o ambiente e obtém o estado inicial
        state = env.reset()  
        
        # Inicializa a ação anterior como None
        prev_action = None  
        
        # Define a variável done como False
        done = False  
        
        # Inicializa listas para armazenar recompensas, ações e logaritmos de probabilidades
        rewards, actions, log_probs = [], [], []  

        # Loop infinito, limitado pela capacidade máxima de um inteiro
        for _ in range(np.iinfo(np.int32).max):  
            
            # Obtém a ação e o log_prob usando a função act
            action, log_prob = act(policy_network, state, prev_action, exploration_rate)  
            
            # Atualiza o estado, obtém a recompensa e verifica se o episódio terminou
            state, reward, done, _ = env.step(action)  
            
            # Atualiza a ação anterior
            prev_action = torch.Tensor(action).unsqueeze(0)  

            # Se o episódio terminou, sai do loop
            if done:  
                break

            # Adiciona a ação à lista de ações
            actions.append(action)  
            
            # Adiciona a recompensa à lista de recompensas
            rewards.append(reward)  
            
            # Adiciona o log_prob à lista de log_probs
            log_probs.append(log_prob)  
        
        # Se estiver em treinamento e houver recompensas
        if train and rewards != []:  
            
            # Calcula recompensas ponderadas pelo fator de desconto
            weighted_rewards = list(accumulate(reversed(rewards), lambda x,y: x * discount_factor + y))[::-1]
            
            # Calcula a perda da política
            policy_loss = dsa_get_policy_loss(weighted_rewards, log_probs)  
            
            # Otimiza a política
            dsa_optimize(optimizer, policy_loss)  

        # Adiciona a soma das recompensas ao histórico
        reward_history.append(sum(rewards))  
        
        # Adiciona a lista de ações ao histórico
        action_history.append(np.array(actions))  
        
        # Atualiza a taxa de exploração
        exploration_rate = max(exploration_rate * exploration_decay, exploration_min)  

    # Retorna o histórico de recompensas e ações
    return np.array(reward_history), np.array(action_history)  

## Loop de Treinamento

Primeiro, obtemos os preços de fechamento e os estados correspondentes.

In [ ]:
# Obtemos os dados
price, states = dsa_get_states()

In [ ]:
# Número de atributos
num_features = states.shape[1]

In [ ]:
# Número de observações prévias
num_prev_obs = 3

In [ ]:
# Número total de atributos
total_num_features = num_features * num_prev_obs

Em seguida, dividimos os estados e preços em conjuntos de treinamento e teste e criamos um ambiente de treinamento e teste.

In [ ]:
train_split = 1/2

In [ ]:
states_train = states[:int(states.shape[0] * train_split)]

In [ ]:
states_test = states[-int(states.shape[0] * (1-train_split)):]

In [ ]:
price_train = price[:int(len(price) * train_split)]

In [ ]:
price_test = price[-int(len(price) * (1-train_split)):]

In [ ]:
std_factor = 0.1

In [ ]:
# Ambiente de treino
ambiente_treino = DSATradeEnvironment(states = states_train, 
                                      prices = price_train, 
                                      num_prev_observations = num_prev_obs, 
                                      std_factor = std_factor)

In [ ]:
# Ambiente de teste
ambiente_teste = DSATradeEnvironment(states = states_test, 
                                     prices = price_test, 
                                     num_prev_observations = num_prev_obs, 
                                     std_factor = std_factor)

Em seguida, definimos a rede com as políticas e a treinamos no ambiente de treinamento usando o algoritmo RL.

In [ ]:
# Cria a política
policy = DSARedeNeural(observation_space = total_num_features, action_space = 1).to(device)

In [ ]:
# Treina o agente
scores, actions = dsa_executa_agente(policy, 
                                     ambiente_treino, 
                                     act = dsa_act_policy, 
                                     alpha = 0.1, 
                                     num_episodes = 100)

> Exibimos recompensa de treinamento ao longo das épocas.

In [ ]:
# Plot
plt.plot(scores, label = 'Recompensa em Treino')
plt.xlabel('Episódios')
plt.ylabel('Reward')
plt.legend()
plt.show()

## Execução do Agente Para Simulação Financeira

> Executaremos o agente no ambiente de teste.

In [ ]:
# Executa o agente
scores, actions = dsa_executa_agente(policy, 
                                     ambiente_teste, 
                                     act = dsa_act_policy, 
                                     alpha = 0.1, 
                                     num_episodes = 10,
                                     train = False)

> Calculamos os retornos cumulativos das ações geradas pelo agente no conjunto de teste.

In [ ]:
# Plot
plt.plot(np.cumprod(actions[0].flatten() * price_test.pct_change().values[1:] + 1), label = 'Retorno do Agente')
plt.xlabel('Tempo')
plt.ylabel('Retorno Acumulado')
plt.legend()
plt.show()

# Fim